# HiRISE Priority-10 Boulder Detection

Download 10 HiRISE Mars images from the PDS, convert JP2 → GeoTIFF, and run YOLOv8 boulder detection on each.

**Sections**
1. Image catalog
2. Download JP2 files
3. Convert JP2 → GeoTIFF
4. CRS sanity check
5. Load YOLO detection model
6. Run detection
7. Summary


In [ ]:
import os, sys, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from tqdm import tqdm

SCRATCH       = Path('/scratch/users/cayleigh')
DATA_DIR      = SCRATCH / 'hirise_priority10'
JP2_DIR       = DATA_DIR / 'jp2'
TIF_DIR       = DATA_DIR / 'tif'
OUT_DIR       = DATA_DIR / 'detections'
for d in [JP2_DIR, TIF_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_WEIGHTS = SCRATCH / 'YOLOv8-BeyondEarth/yolov8_model/yolov8-m-boulder-detection-tmp.pt'
CONFIDENCE    = 0.1
SLICE_SIZE    = 512

print('directories ready')
r = subprocess.run(['df', '-h', str(SCRATCH)], capture_output=True, text=True)
print(r.stdout.strip())


## Part 1 — Image Catalog


In [ ]:
IMAGES = [
    {'obs_id': 'ESP_055714_2270', 'boulder_label': 'rich',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_055700_055799/ESP_055714_2270/ESP_055714_2270_RED.JP2'},
    {'obs_id': 'ESP_054857_2270', 'boulder_label': 'rich',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_054800_054899/ESP_054857_2270/ESP_054857_2270_RED.JP2'},
    {'obs_id': 'ESP_069669_2220', 'boulder_label': 'rich',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_069600_069699/ESP_069669_2220/ESP_069669_2220_RED.JP2'},
    {'obs_id': 'ESP_057469_2215', 'boulder_label': 'rich',    'terrain': 'mesas',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_057400_057499/ESP_057469_2215/ESP_057469_2215_RED.JP2'},
    {'obs_id': 'ESP_071093_2210', 'boulder_label': 'rich',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_071000_071099/ESP_071093_2210/ESP_071093_2210_RED.JP2'},
    {'obs_id': 'ESP_047976_2020', 'boulder_label': 'rich',    'terrain': 'mesas',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_047900_047999/ESP_047976_2020/ESP_047976_2020_RED.JP2'},
    {'obs_id': 'ESP_056165_2200', 'boulder_label': 'poor',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_056100_056199/ESP_056165_2200/ESP_056165_2200_RED.JP2'},
    {'obs_id': 'ESP_075577_2105', 'boulder_label': 'poor',    'terrain': 'plains',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_075500_075599/ESP_075577_2105/ESP_075577_2105_RED.JP2'},
    {'obs_id': 'ESP_039820_1750', 'boulder_label': 'unknown', 'terrain': 'unknown',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_039800_039899/ESP_039820_1750/ESP_039820_1750_RED.JP2'},
    {'obs_id': 'ESP_065711_1545', 'boulder_label': 'unknown', 'terrain': 'unknown',
     'jp2_url': 'https://hirise.lpl.arizona.edu/PDS/RDR/ESP/ORB_065700_065799/ESP_065711_1545/ESP_065711_1545_RED.JP2'},
]
print(f'{len(IMAGES)} images')
pd.DataFrame(IMAGES)[['obs_id', 'boulder_label', 'terrain']]


## Part 2 — Download JP2 Files

HiRISE RED images are typically 1–4 GB compressed. Downloads skip automatically if the file already exists.


In [ ]:
def download_jp2(url, dest, chunk_mb=4):
    if dest.exists():
        print(f'  exists ({dest.stat().st_size/1e9:.2f} GB): {dest.name}')
        return True
    print(f'  downloading: {dest.name}')
    try:
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))
        print(f'  done ({dest.stat().st_size/1e9:.2f} GB): {dest.name}')
        return True
    except Exception as e:
        print(f'  ERROR {dest.name}: {e}')
        dest.unlink(missing_ok=True)
        return False

for img in IMAGES:
    jp2_path = JP2_DIR / f"{img['obs_id']}_RED.JP2"
    ok = download_jp2(img['jp2_url'], jp2_path)
    img['jp2_path'] = jp2_path if ok else None


## Part 3 — Convert JP2 → GeoTIFF

Uses `gdal_translate` with LZW compression and tiling. Output files are typically 5–20 GB. Conversion skips if the GeoTIFF already exists.


In [ ]:
r = subprocess.run(['gdal_translate', '--version'], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())


In [ ]:
def jp2_to_geotiff(jp2_path, tif_path):
    if tif_path.exists():
        print(f'  exists ({tif_path.stat().st_size/1e9:.2f} GB): {tif_path.name}')
        return True
    cmd = [
        'gdal_translate', '-of', 'GTiff',
        '-co', 'COMPRESS=LZW',
        '-co', 'TILED=YES',
        '-co', 'BIGTIFF=YES',
        str(jp2_path), str(tif_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ERROR converting {jp2_path.name}:\n{r.stderr}')
        return False
    print(f'  converted ({tif_path.stat().st_size/1e9:.2f} GB): {tif_path.name}')
    return True

for img in IMAGES:
    if img.get('jp2_path') is None:
        img['tif_path'] = None
        continue
    tif_path = TIF_DIR / f"{img['obs_id']}_RED.tif"
    ok = jp2_to_geotiff(img['jp2_path'], tif_path)
    img['tif_path'] = tif_path if ok else None


## Part 4 — CRS Sanity Check

Verify each GeoTIFF has valid spatial metadata before running detection. If CRS shows `None`, the JP2 lacked embedded projection info — those images will need the LBL file parsed to set CRS manually.


In [ ]:
for img in IMAGES:
    tif = img.get('tif_path')
    if tif is None or not tif.exists():
        print(f"  MISSING: {img['obs_id']}")
        continue
    with rasterio.open(tif) as src:
        crs_str = str(src.crs)[:70] if src.crs else 'NO CRS — needs LBL'
        res_m   = tuple(round(x, 4) for x in src.res)
        print(f"{img['obs_id']}  {src.shape}  res={res_m}  {crs_str}")


## Part 5 — Load YOLO Detection Model


In [ ]:
import typing_extensions
if not hasattr(typing_extensions, 'TypeIs'):
    typing_extensions.TypeIs = typing_extensions.TypeGuard
import torch, torch._utils
if not hasattr(torch, '_utils'):
    torch._utils = sys.modules['torch._utils']

from sahi import AutoDetectionModel
from YOLOv8BeyondEarth.predict import get_sliced_predictionfast
from shptools_BOULDERING import shp

assert MODEL_WEIGHTS.exists(), f'weights not found: {MODEL_WEIGHTS}'

detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=MODEL_WEIGHTS.as_posix(),
    confidence_threshold=CONFIDENCE,
    device='cuda:0',
    image_size=1024,
)
print('model loaded OK')
print(f'GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')


## Part 6 — Run Detection

SAHI sliced inference on each GeoTIFF. Results saved as shapefiles under `detections/<obs_id>/`. Existing detections are skipped.


In [ ]:
for img in IMAGES:
    obs      = img['obs_id']
    tif_path = img.get('tif_path')
    if tif_path is None or not tif_path.exists():
        print(f'SKIP {obs} (no tif)')
        img['shp_path'] = None
        continue

    out_dir = OUT_DIR / obs
    out_dir.mkdir(exist_ok=True)
    shp_out = out_dir / f'{obs}-detections-nms.shp'

    if shp_out.exists():
        gdf = gpd.read_file(shp_out)
        print(f'skip {obs} (exists, {len(gdf)} detections)')
        img['shp_path']     = shp_out
        img['n_detections'] = len(gdf)
        continue

    label = img['boulder_label']
    terrain = img['terrain']
    print(f'\n── {obs} ({label}, {terrain}) ──')
    t0 = time.time()

    result = get_sliced_predictionfast(
        image=str(tif_path),
        detection_model=detection_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        perform_standard_pred=False,
        postprocess_type='NMM',
        postprocess_match_metric='IOS',
        postprocess_match_threshold=0.5,
        verbose=0,
        downscale_pred=False,
    )

    elapsed = time.time() - t0
    n = len(result.object_prediction_list)
    print(f'  {n} detections  ({elapsed:.0f}s)')

    gdf = shp.prediction_to_geodataframe(result, str(tif_path))
    gdf.to_file(shp_out)
    print(f'  saved: {shp_out}')
    img['shp_path']     = shp_out
    img['n_detections'] = len(gdf)
    torch.cuda.empty_cache()


## Part 7 — Summary


In [ ]:
rows = []
for img in IMAGES:
    shp_path = img.get('shp_path')
    n = img.get('n_detections')
    if shp_path and shp_path.exists() and n is None:
        n = len(gpd.read_file(shp_path))
    rows.append({
        'obs_id':        img['obs_id'],
        'boulder_label': img['boulder_label'],
        'terrain':       img['terrain'],
        'n_detections':  n,
        'shp':           str(shp_path) if shp_path else None,
    })

summary = pd.DataFrame(rows)
print(summary[['obs_id', 'boulder_label', 'terrain', 'n_detections']].to_string(index=False))
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f"\nsaved: {OUT_DIR / 'summary.csv'}")
